# TriaGS + GT Depth Supervision 파이프라인

기존 TriaGS에 **GT depth 활용**을 추가한 버전입니다.

### 기존 대비 개선점
1. **Depth back-projection으로 초기 포인트 생성** — 랜덤 → 표면 위 정확한 점
2. **Depth supervision loss** — 렌더링 depth vs GT depth 직접 비교
3. **Depth > 0 → 오브젝트 마스크** — 배경 자동 분리

---

| 단계 | 설명 |
|------|------|
| **0** | 환경 설정 |
| **1** | RGB + Depth 데이터 다운로드 & 변환 |
| **2** | 학습 (depth supervision 포함) |
| **3** | 메쉬 추출 |
| **4** | 렌더링 + 평가 |
| **5** | 시각화 |
| **6** | 결과 저장 |

---
## 0. 환경 설정

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("GPU 필요!")

In [ ]:
import os

WORK_DIR = "/content/triags"
DATA_DIR = "/content/data"
OUTPUT_DIR = "/content/output"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
if not os.path.exists(WORK_DIR):
    !git clone https://github.com/BAEJUNHAK/triags {WORK_DIR}
os.chdir(WORK_DIR)

In [ ]:
!pip install -q open3d trimesh mediapy scikit-image opencv-python plyfile tqdm tensorboard gdown
!pip install -q submodules/diff-gaussian-rasterization
!pip install -q git+https://github.com/camenduru/simple-knn

---
## 1. 데이터 다운로드 & 변환 (RGB + Depth)

In [ ]:
import gdown

# RGB + Camera
RGB_ZIP = f"{DATA_DIR}/dataset.zip"
if not os.path.exists(RGB_ZIP):
    gdown.download("https://drive.google.com/uc?id=1dnj1s-mqIuS6OcdSr5CczBr9u8yBzUUB", RGB_ZIP, quiet=False)

# Depth
DEPTH_ZIP = f"{DATA_DIR}/depth.zip"
if not os.path.exists(DEPTH_ZIP):
    gdown.download("https://drive.google.com/uc?id=1KmXCzBYv_mkPmZnWka1ThCZSNHTyivKQ", DEPTH_ZIP, quiet=False)

# 압축 해제
!unzip -q -o {RGB_ZIP} -d {DATA_DIR}/raw_rgb
!unzip -q -o {DEPTH_ZIP} -d {DATA_DIR}/raw_depth
print("다운로드 완료")

In [ ]:
import glob

def find_data_root(base_dir, prefix):
    for root, dirs, files in os.walk(base_dir):
        if any(f.startswith(prefix) for f in files):
            return root
    return base_dir

RGB_DATA_DIR = find_data_root(f"{DATA_DIR}/raw_rgb", "calib_")
DEPTH_DATA_DIR = find_data_root(f"{DATA_DIR}/raw_depth", "depth_raw_")

print(f"RGB 폴더:   {RGB_DATA_DIR}")
print(f"Depth 폴더: {DEPTH_DATA_DIR}")
print(f"RGB 파일:   {len(glob.glob(f'{RGB_DATA_DIR}/rgb_*.png'))}개")
print(f"Depth 파일: {len(glob.glob(f'{DEPTH_DATA_DIR}/depth_raw_*.png'))}개")
print(f"Calib 파일: {len(glob.glob(f'{RGB_DATA_DIR}/calib_*.ini'))}개")

In [ ]:
#@title 데이터 설정 {display-mode: "form"}

NUM_TRAIN = 160  #@param {type:"integer"}
NUM_TEST = 40  #@param {type:"integer"}
DEPTH_SCALE = 0.01  #@param {type:"number"}
#@markdown ↑ uint16 raw값 × DEPTH_SCALE = 실제 거리

NUM_TOTAL = NUM_TRAIN + NUM_TEST
print(f"Train: {NUM_TRAIN}, Test: {NUM_TEST}, 총: {NUM_TOTAL}")

In [ ]:
import numpy as np
import cv2
from pathlib import Path
import shutil

SCENE_DIR = f"{DATA_DIR}/scene"

def parse_calib(filepath):
    config = {}
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('['):
                continue
            key, value = line.split('=', 1)
            config[key] = value
    k_vals = list(map(float, config['k_matrix'].split(':')[-1].split(',')))
    K = np.array(k_vals).reshape(3, 3)
    r_vals = list(map(float, config['r_matrix'].split(':')[-1].split(',')))
    R = np.array(r_vals).reshape(3, 3)
    t_vals = list(map(float, config['t_vector'].split(':')[-1].split(',')))
    T = np.array(t_vals)
    return K, R, T, int(config['width']), int(config['height'])

def rotmat2qvec(R):
    Rxx, Ryx, Rzx, Rxy, Ryy, Rzy, Rxz, Ryz, Rzz = R.flat
    K = np.array([
        [Rxx - Ryy - Rzz, 0, 0, 0],
        [Ryx + Rxy, Ryy - Rxx - Rzz, 0, 0],
        [Rzx + Rxz, Rzy + Ryz, Rzz - Rxx - Ryy, 0],
        [Ryz - Rzy, Rzx - Rxz, Rxy - Ryx, Rxx + Ryy + Rzz]]) / 3.0
    eigvals, eigvecs = np.linalg.eigh(K)
    qvec = eigvecs[[3, 0, 1, 2], np.argmax(eigvals)]
    if qvec[0] < 0:
        qvec *= -1
    return qvec

# 파일 매칭
def get_index(filepath):
    return Path(filepath).stem.split('_')[-1]

calib_dict = {get_index(f): f for f in sorted(glob.glob(f"{RGB_DATA_DIR}/calib_*.ini"))}
rgb_dict = {get_index(f): f for f in sorted(glob.glob(f"{RGB_DATA_DIR}/rgb_*.png"))}
depth_dict = {get_index(f): f for f in sorted(glob.glob(f"{DEPTH_DATA_DIR}/depth_raw_*.png"))}

# 세 파일 모두 있는 인덱스만
all_indices = sorted(set(calib_dict.keys()) & set(rgb_dict.keys()) & set(depth_dict.keys()))
N_ALL = len(all_indices)
print(f"RGB+Depth+Calib 쌍: {N_ALL}개")

# 균일 샘플링 + 인터리빙 분할
if NUM_TOTAL < N_ALL:
    sample_idx = np.linspace(0, N_ALL - 1, NUM_TOTAL, dtype=int)
    selected_indices = [all_indices[i] for i in sample_idx]
else:
    selected_indices = all_indices

stride = max(1, len(selected_indices) // NUM_TEST)
train_indices = []
test_indices = []
for i, idx in enumerate(selected_indices):
    if (i + 1) % stride == 0:
        test_indices.append(idx)
    else:
        train_indices.append(idx)

print(f"Train: {len(train_indices)}, Test: {len(test_indices)}")

In [ ]:
# ============================================================
# COLMAP 변환 + depth back-projection 초기 포인트 생성
# ============================================================
if os.path.exists(SCENE_DIR):
    shutil.rmtree(SCENE_DIR)

images_dir = os.path.join(SCENE_DIR, "images")
sparse_dir = os.path.join(SCENE_DIR, "sparse")
depth_out_dir = os.path.join(SCENE_DIR, "depth")  # TriaGS가 읽을 depth 폴더
test_images_dir = os.path.join(SCENE_DIR, "test_images")
os.makedirs(images_dir, exist_ok=True)
os.makedirs(sparse_dir, exist_ok=True)
os.makedirs(depth_out_dir, exist_ok=True)
os.makedirs(test_images_dir, exist_ok=True)

# cameras.txt
K0, _, _, W0, H0 = parse_calib(calib_dict[train_indices[0]])
fx, fy, cx, cy = K0[0, 0], K0[1, 1], K0[0, 2], K0[1, 2]

with open(os.path.join(sparse_dir, "cameras.txt"), 'w') as f:
    f.write("# Camera list with one line of data per camera:\n")
    f.write("# CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[]\n")
    f.write(f"# Number of cameras: 1\n")
    f.write(f"1 PINHOLE {W0} {H0} {fx} {fy} {cx} {cy}\n")

# images.txt + depth 복사 + 이미지 링크
camera_positions = []
all_backproj_points = []
all_backproj_colors = []

# depth back-projection용 샘플 수 (전체 하면 너무 많으므로 뷰당 제한)
POINTS_PER_VIEW = 2000

with open(os.path.join(sparse_dir, "images.txt"), 'w') as f:
    f.write("# Image list with two lines of data per image:\n")
    f.write("# IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME\n")
    f.write("# POINTS2D[] as (X, Y, POINT3D_ID)\n")
    f.write(f"# Number of images: {len(train_indices)}\n")

    for img_id, idx in enumerate(train_indices, start=1):
        K, R, T, W, H = parse_calib(calib_dict[idx])
        qvec = rotmat2qvec(R)
        cam_pos = -R.T @ T
        camera_positions.append(cam_pos)

        # 이미지 심볼릭 링크
        img_name = f"rgb_{idx}.png"
        os.symlink(os.path.abspath(rgb_dict[idx]), os.path.join(images_dir, img_name))

        # depth 파일 복사 (이름 매칭: rgb_0000 → depth_raw_0000)
        depth_src = depth_dict.get(idx)
        if depth_src:
            depth_dst_name = f"depth_raw_rgb_{idx}.png"  # load_gt_depth가 찾을 수 있는 패턴
            os.symlink(os.path.abspath(depth_src), os.path.join(depth_out_dir, depth_dst_name))

        f.write(f"{img_id} {qvec[0]} {qvec[1]} {qvec[2]} {qvec[3]} {T[0]} {T[1]} {T[2]} 1 {img_name}\n")
        f.write("\n")

        # ---- Depth back-projection: 표면 위 3D 점 생성 ----
        if depth_src:
            depth_raw = cv2.imread(depth_src, cv2.IMREAD_UNCHANGED)
            depth_m = depth_raw.astype(np.float32) * DEPTH_SCALE

            # 유효 픽셀 (depth > 0)
            valid_mask = depth_m > 0
            ys, xs = np.where(valid_mask)

            if len(xs) > POINTS_PER_VIEW:
                choice = np.random.choice(len(xs), POINTS_PER_VIEW, replace=False)
                xs, ys = xs[choice], ys[choice]

            depths = depth_m[ys, xs]

            # 카메라 좌표계로 역투영: X_cam = K^-1 @ [u, v, 1] * depth
            pts_cam = np.stack([
                (xs - cx) / fx * depths,
                (ys - cy) / fy * depths,
                depths
            ], axis=1)  # (N, 3)

            # 월드 좌표로 변환: X_world = R^T @ (X_cam - T)
            pts_world = (R.T @ (pts_cam.T - T[:, None])).T  # (N, 3)

            all_backproj_points.append(pts_world)

            # RGB 색상 가져오기
            rgb_img = cv2.imread(rgb_dict[idx])
            rgb_img = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2RGB)
            colors = rgb_img[ys, xs]  # (N, 3) uint8
            all_backproj_colors.append(colors)

# test 이미지 복사
for idx in test_indices:
    os.symlink(os.path.abspath(rgb_dict[idx]), os.path.join(test_images_dir, f"rgb_{idx}.png"))

camera_positions = np.array(camera_positions)
print(f"[OK] cameras.txt, images.txt (Train {len(train_indices)}장)")
print(f"[OK] depth/ ({len(os.listdir(depth_out_dir))}개 depth 파일)")
print(f"[OK] test_images/ (Test {len(test_indices)}장)")

In [ ]:
# ============================================================
# points3D.ply: depth back-projection 기반 초기 포인트
# ============================================================
from plyfile import PlyData, PlyElement

all_pts = np.concatenate(all_backproj_points, axis=0).astype(np.float32)
all_rgb = np.concatenate(all_backproj_colors, axis=0).astype(np.uint8)

print(f"Back-projected 포인트: {len(all_pts):,}개")

# 너무 많으면 서브샘플링 (GPU 메모리 고려)
MAX_INIT_POINTS = 200000
if len(all_pts) > MAX_INIT_POINTS:
    choice = np.random.choice(len(all_pts), MAX_INIT_POINTS, replace=False)
    all_pts = all_pts[choice]
    all_rgb = all_rgb[choice]
    print(f"서브샘플링 → {len(all_pts):,}개")

# 이상치 제거 (중심에서 너무 먼 점)
center = np.median(all_pts, axis=0)
dists = np.linalg.norm(all_pts - center, axis=1)
threshold = np.percentile(dists, 99)
valid = dists < threshold
all_pts = all_pts[valid]
all_rgb = all_rgb[valid]
print(f"이상치 제거 후: {len(all_pts):,}개")
print(f"포인트 범위: {all_pts.min(axis=0)} ~ {all_pts.max(axis=0)}")

# PLY 저장
with open(os.path.join(sparse_dir, "points3D.txt"), 'w') as f:
    f.write("# 3D point list\n")

ply_path = os.path.join(sparse_dir, "points3D.ply")
dtype = [('x','f4'),('y','f4'),('z','f4'),('nx','f4'),('ny','f4'),('nz','f4'),('red','u1'),('green','u1'),('blue','u1')]
normals = np.zeros_like(all_pts, dtype=np.float32)
elements = np.empty(len(all_pts), dtype=dtype)
elements[:] = list(map(tuple, np.concatenate([all_pts, normals, all_rgb], axis=1)))
PlyData([PlyElement.describe(elements, 'vertex')]).write(ply_path)

print(f"\n[OK] points3D.ply ({len(all_pts):,} points, depth back-projection)")
print(f"     vs 이전: 랜덤 100,000점")

In [ ]:
# 초기 포인트 시각화
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(12, 5))

# 3D 포인트 클라우드
ax1 = fig.add_subplot(121, projection='3d')
step = max(1, len(all_pts) // 5000)
ax1.scatter(all_pts[::step, 0], all_pts[::step, 1], all_pts[::step, 2],
            c=all_rgb[::step] / 255.0, s=1, alpha=0.5)
ax1.set_title(f'초기 포인트 ({len(all_pts):,}개, depth back-proj)')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

# 카메라 위치
ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(camera_positions[::2, 0], camera_positions[::2, 1], camera_positions[::2, 2],
            c='blue', s=5, alpha=0.5, label='Cameras')
ax2.scatter(all_pts[::step*5, 0], all_pts[::step*5, 1], all_pts[::step*5, 2],
            c='gray', s=0.5, alpha=0.2, label='Points')
ax2.legend()
ax2.set_title('카메라 + 포인트 배치')

plt.tight_layout()
plt.show()

---
## 2. 학습 (Depth Supervision 포함)

In [ ]:
#@title 학습 파라미터 {display-mode: "form"}

ITERATIONS = 30000  #@param {type:"integer"}
RESOLUTION = 2  #@param [1, 2, 4] {type:"raw"}
USE_APPEARANCE = True  #@param {type:"boolean"}

TGPC_WEIGHT = 0.1  #@param {type:"number"}
TGPC_NEIGHBORS = 12  #@param {type:"integer"}
MULTI_VIEW_NUM = 8  #@param {type:"integer"}
MULTI_VIEW_MAX_ANGLE = 30  #@param {type:"integer"}
MULTI_VIEW_MIN_DIS = 1.0  #@param {type:"number"}
MULTI_VIEW_MAX_DIS = 200.0  #@param {type:"number"}

LAMBDA_DSSIM = 0.2  #@param {type:"number"}
LAMBDA_DEPTH_NORMAL = 0.05  #@param {type:"number"}
LAMBDA_GEO = 0.03  #@param {type:"number"}
LAMBDA_NCC = 0.15  #@param {type:"number"}

#@markdown ### Depth Supervision (NEW)
LAMBDA_GT_DEPTH = 0.1  #@param {type:"number"}

SOURCE_PATH = SCENE_DIR
MODEL_PATH = f"{OUTPUT_DIR}/custom_scene_depth"

print(f"Source: {SOURCE_PATH}")
print(f"Output: {MODEL_PATH}")
print(f"Depth supervision: lambda={LAMBDA_GT_DEPTH}")

In [ ]:
os.chdir(WORK_DIR)

cmd = f"""python train.py \
    -s {SOURCE_PATH} \
    -m {MODEL_PATH} \
    -r {RESOLUTION} \
    --iterations {ITERATIONS} \
    --use_gt_depth \
    --depth_scale {DEPTH_SCALE} \
    --lambda_gt_depth {LAMBDA_GT_DEPTH} \
    --lambda_dssim {LAMBDA_DSSIM} \
    --lambda_depth_normal {LAMBDA_DEPTH_NORMAL} \
    --multi_view_geo_weight {LAMBDA_GEO} \
    --multi_view_ncc_weight {LAMBDA_NCC} \
    --tgpc_loss_weight {TGPC_WEIGHT} \
    --tgpc_num_neighbors {TGPC_NEIGHBORS} \
    --multi_view_num {MULTI_VIEW_NUM} \
    --multi_view_max_angle {MULTI_VIEW_MAX_ANGLE} \
    --multi_view_min_dis {MULTI_VIEW_MIN_DIS} \
    --multi_view_max_dis {MULTI_VIEW_MAX_DIS} \
    --test_iterations 7000 15000 30000 \
    --save_iterations 7000 30000"""

if USE_APPEARANCE:
    cmd += " --use_decoupled_appearance"

print(cmd)

In [ ]:
!{cmd}

---
## 3. 메쉬 추출

In [ ]:
os.chdir(WORK_DIR)
!python mesh_extract.py -s {SOURCE_PATH} -m {MODEL_PATH}

In [ ]:
import trimesh
for mf in glob.glob(f"{MODEL_PATH}/**/fuse_*_post.ply", recursive=True) + \
          glob.glob(f"{MODEL_PATH}/**/recon.ply", recursive=True):
    mesh = trimesh.load(mf)
    print(f"{mf}")
    print(f"  Vertices: {len(mesh.vertices):,}, Faces: {len(mesh.faces):,}")

---
## 4. 렌더링 + 평가

In [ ]:
import torch
import json
import numpy as np
from PIL import Image
import torchvision.transforms.functional as tf
from tqdm import tqdm

os.chdir(WORK_DIR)

from scene import GaussianModel
from gaussian_renderer import render as gs_render
from arguments import ModelParams, PipelineParams
from argparse import ArgumentParser
from scene.cameras import Camera as GSCamera
from utils.graphics_utils import focal2fov
from utils.loss_utils import ssim
from utils.image_utils import psnr
from lpipsPyTorch import lpips

parser = ArgumentParser()
lp = ModelParams(parser)
pp = PipelineParams(parser)
args = parser.parse_args(['-s', SOURCE_PATH, '-m', MODEL_PATH])
dataset = lp.extract(args)
pipe = pp.extract(args)

gaussians = GaussianModel(dataset.sh_degree)
from utils.system_utils import searchForMaxIteration
loaded_iter = searchForMaxIteration(os.path.join(MODEL_PATH, "point_cloud"))
gaussians.load_ply(os.path.join(MODEL_PATH, "point_cloud", f"iteration_{loaded_iter}", "point_cloud.ply"))
print(f"모델 로드: iteration {loaded_iter}, {gaussians.get_xyz.shape[0]:,} Gaussians")

bg_color = torch.tensor([0, 0, 0], dtype=torch.float32, device="cuda")
kernel_size = dataset.kernel_size

test_render_dir = os.path.join(MODEL_PATH, "test", f"ours_{loaded_iter}", "renders")
test_gt_dir = os.path.join(MODEL_PATH, "test", f"ours_{loaded_iter}", "gt")
os.makedirs(test_render_dir, exist_ok=True)
os.makedirs(test_gt_dir, exist_ok=True)

psnrs, ssims, lpipss = [], [], []

for ti, idx in enumerate(tqdm(test_indices, desc="Test rendering")):
    K, R, T, W, H = parse_calib(calib_dict[idx])
    scale = RESOLUTION
    W_s, H_s = W // scale, H // scale
    fx_s, fy_s = K[0,0] / scale, K[1,1] / scale
    FovX = focal2fov(fx_s, W_s)
    FovY = focal2fov(fy_s, H_s)

    gt_img = Image.open(rgb_dict[idx]).resize((W_s, H_s), Image.LANCZOS)
    gt_tensor = tf.to_tensor(gt_img).cuda()

    cam = GSCamera(
        colmap_id=ti, R=R.T, T=T,
        FoVx=FovX, FoVy=FovY,
        image=gt_tensor, gt_alpha_mask=None,
        image_name=f"test_{idx}", image_path=rgb_dict[idx], uid=ti,
        data_device="cuda"
    )

    with torch.no_grad():
        out = gs_render(cam, gaussians, pipe, bg_color, kernel_size=kernel_size)
        rendered = out["render"].clamp(0.0, 1.0)

    psnrs.append(psnr(rendered.unsqueeze(0), gt_tensor.unsqueeze(0)).item())
    ssims.append(ssim(rendered.unsqueeze(0), gt_tensor.unsqueeze(0)).item())
    lpipss.append(lpips(rendered.unsqueeze(0), gt_tensor.unsqueeze(0), net_type='vgg').item())

    import torchvision
    torchvision.utils.save_image(rendered, os.path.join(test_render_dir, f"test_{idx}.png"))
    torchvision.utils.save_image(gt_tensor, os.path.join(test_gt_dir, f"test_{idx}.png"))

print(f"\n{'='*50}")
print(f"Test 평가 결과 ({len(test_indices)}장)")
print(f"{'='*50}")
print(f"  PSNR:  {np.mean(psnrs):.3f} dB")
print(f"  SSIM:  {np.mean(ssims):.4f}")
print(f"  LPIPS: {np.mean(lpipss):.4f}")

test_results = {
    "test_metrics": {
        "PSNR": float(np.mean(psnrs)),
        "SSIM": float(np.mean(ssims)),
        "LPIPS": float(np.mean(lpipss)),
        "num_test_views": len(test_indices),
        "num_train_views": len(train_indices),
        "iteration": loaded_iter,
        "depth_supervision": True,
        "lambda_gt_depth": LAMBDA_GT_DEPTH,
    }
}
with open(os.path.join(MODEL_PATH, "test_results.json"), 'w') as f:
    json.dump(test_results, f, indent=2)
print(f"저장: {MODEL_PATH}/test_results.json")

---
## 5. 시각화

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

render_files = sorted(glob.glob(f"{test_render_dir}/*.png"))[:6]
if render_files:
    n = len(render_files)
    fig, axes = plt.subplots(n, 2, figsize=(12, 5*n))
    if n == 1: axes = axes[np.newaxis, :]
    axes[0, 0].set_title("Rendered", fontsize=14, fontweight='bold')
    axes[0, 1].set_title("Ground Truth", fontsize=14, fontweight='bold')
    for i, rf in enumerate(render_files):
        fname = os.path.basename(rf)
        axes[i, 0].imshow(Image.open(rf)); axes[i, 0].set_ylabel(fname); axes[i, 0].axis('off')
        gt = os.path.join(test_gt_dir, fname)
        if os.path.exists(gt): axes[i, 1].imshow(Image.open(gt))
        axes[i, 1].axis('off')
    plt.suptitle(f"Test Views (PSNR: {np.mean(psnrs):.2f} dB)", fontsize=16)
    plt.tight_layout()
    plt.show()

---
## 6. 결과 저장

In [ ]:
import json, zipfile

experiment_config = {
    "dataset": {
        "total_images": N_ALL,
        "num_train": len(train_indices),
        "num_test": len(test_indices),
        "resolution": RESOLUTION,
        "depth_supervision": True,
        "depth_scale": DEPTH_SCALE,
        "init_points": "depth_backprojection",
    },
    "training": {
        "iterations": ITERATIONS,
        "lambda_gt_depth": LAMBDA_GT_DEPTH,
        "tgpc_weight": TGPC_WEIGHT,
        "lambda_dssim": LAMBDA_DSSIM,
        "lambda_depth_normal": LAMBDA_DEPTH_NORMAL,
        "lambda_geo": LAMBDA_GEO,
        "lambda_ncc": LAMBDA_NCC,
    },
    "test_metrics": test_results["test_metrics"],
}

with open(os.path.join(MODEL_PATH, "experiment_config.json"), 'w') as f:
    json.dump(experiment_config, f, indent=2)

ZIP_RESULT = f"{OUTPUT_DIR}/triags_depth_results.zip"
with zipfile.ZipFile(ZIP_RESULT, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(MODEL_PATH):
        if 'point_cloud' in root:
            continue
        for file in files:
            if file.endswith('.pth'):
                continue
            full_path = os.path.join(root, file)
            zf.write(full_path, os.path.relpath(full_path, OUTPUT_DIR))

print(f"zip: {ZIP_RESULT} ({os.path.getsize(ZIP_RESULT)/1e6:.1f} MB)")

try:
    from google.colab import files
    files.download(ZIP_RESULT)
except ImportError:
    print(f"로컬: {ZIP_RESULT}")

In [ ]:
SAVE_TO_DRIVE = False  #@param {type:"boolean"}

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = "/content/drive/MyDrive/triags_results"
    os.makedirs(DRIVE_DIR, exist_ok=True)
    shutil.copy2(ZIP_RESULT, DRIVE_DIR)
    print(f"저장 완료: {DRIVE_DIR}")
else:
    print("Drive 저장 건너뜀")